# 第 0 章 · Agent 框架全景与选型对比

> 本章是整套教程的**地图**。你将了解：
> 1. 什么是 AI Agent，为什么需要"框架"来开发它；
> 2. Google ADK 与 LangChain/LangGraph 的定位与设计哲学；
> 3. 与 CrewAI、AutoGen、Semantic Kernel 等主流框架的横向对比；
> 4. 一张可以带走的**选型决策树**。

---

## 1. 什么是 AI Agent？

一个 **Agent（智能体）** = 大语言模型（LLM）+ 目标 + 工具 + 自主循环。

与普通"问答机器人"的本质区别在于：Agent 不只是"问一句答一句"，而是围绕一个目标，**自主地进行多轮"思考 → 行动 → 观察 → 再思考"的循环**，直到任务完成。

```mermaid
flowchart LR
    U(["🧑 用户目标<br/>帮我规划北京三日游"]) --> A{{"🧠 LLM<br/>推理与决策"}}
    A -->|"决定调用工具"| T["🔧 工具 Tools<br/>搜索 / 计算器 / API / 数据库"]
    T -->|"返回观察结果"| A
    A -->|"任务完成，输出答案"| R(["✅ 最终结果"])
    style A fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style T fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

这个循环在学术界常被称为 **ReAct（Reasoning + Acting）** 模式。几乎所有 Agent 框架的核心，都是在工程化地实现和管控这个循环。

---

## 2. 为什么需要"框架"？

理论上，用几十行 Python + 一个 `while` 循环就能手写一个 Agent。但真实项目中你会立刻遇到一堆工程问题：

| 工程难题 | 徒手开发的痛点 | 框架提供的解法 |
|---|---|---|
| **多模型接入** | 每家 LLM 的 API 协议都不一样 | 统一抽象层，一行代码切换模型 |
| **工具调用** | 手写 JSON Schema、解析函数调用参数很繁琐 | 装饰器/函数签名自动生成工具描述 |
| **流程编排** | 循环、分支、并行、子任务委派越写越乱 | 声明式编排（工作流 Agent / 状态图） |
| **上下文与记忆** | 对话历史、中间状态、长期记忆全要自己管 | Session / State / Memory 内建机制 |
| **可观测性** | Agent "想错了"时无从排查 | Trace、回调、可视化调试 UI |
| **部署运维** | 从脚本到生产服务隔着一道鸿沟 | 评估框架、部署工具链、托管服务 |

> 💡 **一句话总结**：框架卖的不是"能否做出 Agent"，而是**把 Agent 从 demo 推进到生产的工程确定性**。

---

## 3. 两大主角登场

### 3.1 Google ADK（Agent Development Kit）

- **出身**：Google 于 2025 年 4 月开源，与 Gemini、Vertex AI 深度同源，但**模型中立**（内置 LiteLLM 适配层，可接 DeepSeek / OpenAI / Claude 等 100+ 模型）。
- **设计哲学**：**"Agent 是一等公民"**。框架围绕 `Agent` 这个核心抽象展开，层级化多智能体（`sub_agents`）是语言级内建概念。
- **杀手锏**：开发体验极佳（自带 `adk web` 可视化调试 UI）、多智能体编排声明式、与 Google Cloud 生产链路（Agent Engine）打通。

### 3.2 LangChain / LangGraph

- **出身**：LangChain 是 2022 年底爆发的 LLM 应用框架，几乎是"LLM 编程"的代名词；**LangGraph** 是其团队于 2024 年推出的底层**图编排引擎**，如今已成为官方推荐的 Agent 构建方式（LangChain 1.x 的 `create_agent` 底层就是 LangGraph）。
- **设计哲学**：**"一切皆图（Graph）"**。Agent 被建模为状态机：状态（State）在节点（Node）间流动，由边（Edge）控制流转。控制流对开发者完全透明、可定制。
- **杀手锏**：生态极其庞大（几百个集成包）、LangSmith 观测平台成熟、对复杂控制流（循环、人机协同、持久化、时间旅行）的支持业内最强。

```mermaid
flowchart TB
    subgraph G["Google 阵营"]
        ADK["ADK<br/>Agent 一等公民<br/>声明式多智能体"] --> GEM["Gemini / Vertex AI<br/>（也可用 LiteLLM 接任意模型）"]
        ADK --> AE["Agent Engine<br/>托管部署"]
    end
    subgraph L["LangChain 阵营"]
        LC["LangChain 1.x<br/>create_agent / 集成生态"] --> LG["LangGraph<br/>状态图编排引擎"]
        LG --> LS["LangSmith<br/>观测与评估"]
        LG --> LP["LangGraph Platform<br/>托管部署"]
    end
    style ADK fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style LG fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

---

## 4. 主流框架横向对比

下表从**教学中最常被问到**的维度，对当前主流 Agent 框架做一次全景扫描（💚=强，💛=中，🤍=弱）：

| 框架 | 出品方 | 核心范式 | 多智能体 | 学习曲线 | 生态集成 | 生产化 | 一句话点评 |
|---|---|---|---|---|---|---|---|
| **Google ADK** | Google | Agent 层级 + 工作流 Agent | 💚 声明式 | 💛 平缓 | 💛 成长中 | 💚 Agent Engine | 多智能体开发体验最顺滑的新贵 |
| **LangChain + LangGraph** | LangChain Inc. | 状态图（StateGraph） | 💚 图内建 | ❤️ 偏陡 | 💚 最庞大 | 💚 LangSmith/Platform | 控制自由度与生态的天花板 |
| **CrewAI** | CrewAI Inc. | 角色扮演（Role-based） | 💚 角色协作 | 💚 平缓 | 💛 | 💛 | "组队"概念直觉，上手最快 |
| **AutoGen / AG2** | Microsoft 研究院 | 对话式多智能体 | 💚 对话驱动 | 💛 | 💛 | 💛 | 学术味浓，Agent 间"聊天"解题 |
| **Semantic Kernel** | Microsoft | 插件/规划器 | 💛 | 💛 | 💛 | 💚 Azure | 企业级 .NET/Python 双栈 |
| **OpenAI Agents SDK** | OpenAI | 轻量 Agent + Handoff | 💛 Handoff | 💚 极简 | 🤍 OpenAI 系 | 💛 | 极简主义，绑定 OpenAI 生态 |
| **LlamaIndex** | LlamaIndex Inc. | 数据索引 + Agent | 💛 | 💛 | 💚 数据连接器 | 💛 | RAG 起家，数据 ingestion 最强 |
| **Dify / Coze（扣子）** | 字节等 | 低代码画布 | 💛 | 💚 无需编码 | 💚 插件市场 | 💚 开箱即用 | 非程序员友好的低代码平台 |

> ⚠️ **时效性说明**：Agent 框架领域演进极快，上表反映 2025-2026 年的格局。学习时建议聚焦**不变的底层概念**（工具调用、状态管理、编排模式），它们在任何框架中都通用。

---

## 5. ADK vs LangGraph：设计哲学的一对根本差异

本教程最关心的对比，可以浓缩为一句话：

> **ADK 把"多智能体系统"当作语言内建概念；LangGraph 把"一切计算"都降为图上的状态流转。**

| 维度 | Google ADK | LangChain / LangGraph |
|---|---|---|
| 核心抽象 | `Agent`（含 `sub_agents` 层级树） | `State` + `Node` + `Edge` 构成的图 |
| 编排方式 | 声明式：Sequential / Parallel / Loop 工作流 Agent | 显式画图：节点与条件边，循环即"回边" |
| 控制权 | 框架托管较多，约定优于配置 | 开发者掌控每一次状态迁移 |
| 典型心智模型 | "我在组建一个公司，招聘不同岗位的 Agent" | "我在设计一个状态机/电路图" |
| 上手速度 | 快，第一个多智能体系统几十行 | 慢，需要先理解 State/Reducer/Checkpointer |
| 复杂控制流 | 够用（Loop + 回调 + 转移工具） | 天花板极高（可中断、可回放、可分叉） |
| 调试观测 | `adk web` 自带事件流可视化 | LangSmith Trace，颗粒度到每次 LLM 调用 |
| 模型中立性 | Gemini 原生 + LiteLLM 接任意模型 | 天然模型中立，数百个模型集成 |

```mermaid
flowchart LR
    subgraph ADKW["ADK 的世界观"]
        A1["Root Agent"] --> A2["子 Agent A"]
        A1 --> A3["子 Agent B"]
        A3 --> A4["子 Agent C"]
    end
    subgraph LGW["LangGraph 的世界观"]
        S1((State)) --> N1["节点1"]
        N1 --> S2((State))
        S2 --> N2["节点2"]
        N2 -->|条件边| N1
        N2 --> S3((END))
    end
    style A1 fill:#e8f0fe,stroke:#4285f4
    style N1 fill:#e6f4ea,stroke:#34a853
    style N2 fill:#e6f4ea,stroke:#34a853
```

两种世界观**没有高下之分**：ADK 用约定换速度，LangGraph 用显式换控制。后文每一条线路学完后，你都能亲身体会这种差异。

---

## 6. 选型决策树

```mermaid
flowchart TD
    Q0{"你的团队/项目最看重什么？"}
    Q0 -->|"快速搭建多智能体 demo<br/>平滑的开发调试体验"| P1["✅ 选 ADK<br/>（尤其已有 Google Cloud 基建）"]
    Q0 -->|"极致的控制流定制<br/>复杂的循环/中断/回放"| P2["✅ 选 LangGraph"]
    Q0 -->|"已有大量 LangChain 代码或集成需求"| P2
    Q0 -->|"非技术人员搭建、低代码"| P3["考虑 Dify / Coze"]
    Q0 -->|"角色扮演式的业务模拟"| P4["考虑 CrewAI"]
    Q0 -->|".NET 技术栈 / 微软生态"| P5["考虑 Semantic Kernel"]
    P1 --> N1["💡 两者底层概念相通：<br/>学好任意一个，另一个只需 1-2 天迁移"]
    P2 --> N1
    style P1 fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style P2 fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

**务实的建议**：
1. 原型阶段两者都试试（本教程的代码示例都使用同一个 DeepSeek 模型，方便你逐行对照）；
2. 概念上先学 **ADK 的"Agent 层级"**，再学 **LangGraph 的"状态图"** ——前者符合直觉，后者揭示本质；
3. 生产决策看**团队已有基建**：GCP 用户偏 ADK，需要复杂工作流 + 成熟观测的团队偏 LangGraph + LangSmith。

---

## 7. 本教程结构与代码约定

- `google-adk/` 六章：从第一个 Agent 到多智能体、记忆、回调与部署；
- `langchain-langgraph/` 六章：从模型接入到 LCEL、Agent、LangGraph 图、持久化与人机协同；
- 所有示例统一使用 **DeepSeek `deepseek-chat`** 模型，通过环境变量 `DEEPSEEK_API_KEY` 鉴权；
- 每章末尾设有「**对照阅读**」小节，指出另一框架中对应的章节与概念映射。

下面这个单元格做最后的**环境自检**，通过后就可以正式开始两条线路的学习了。


In [1]:
import os, sys

print("Python:", sys.version.split()[0])

# 1) 检查 API Key
assert os.environ.get("DEEPSEEK_API_KEY"), "❌ 未检测到环境变量 DEEPSEEK_API_KEY，请先按 README 配置"
print("✅ DEEPSEEK_API_KEY 已配置（长度 %d）" % len(os.environ["DEEPSEEK_API_KEY"]))

# 2) 检查两个框架的版本
import google.adk, langchain, langgraph, litellm
import importlib.metadata as im
print("✅ google-adk :", im.version("google-adk"))
print("✅ langchain  :", im.version("langchain"))
print("✅ langgraph  :", im.version("langgraph"))
print("✅ litellm    :", im.version("litellm"))
print("\n🎉 环境就绪！建议从 google-adk/01 或 langchain-langgraph/01 开始。")


Python: 3.12.3
✅ DEEPSEEK_API_KEY 已配置（长度 35）


✅ google-adk : 2.6.3
✅ langchain  : 1.2.15
✅ langgraph  : 1.2.11
✅ litellm    : 1.96.2

🎉 环境就绪！建议从 google-adk/01 或 langchain-langgraph/01 开始。


---

### 📌 本章要点回顾

- Agent = LLM + 目标 + 工具 + **自主循环**；框架的价值在于工程确定性；
- ADK：**Agent 一等公民**，声明式多智能体，开发体验顺滑；
- LangChain/LangGraph：**一切皆状态图**，控制自由度与生态最强；
- 选型先看团队基建与控制流复杂度，两个框架的概念高度互通。

> ➡️ 下一站：[google-adk/01-初识ADK-快速上手](google-adk/01-初识ADK-快速上手.ipynb) 或 [langchain-langgraph/01-LangChain生态与模型接入](langchain-langgraph/01-LangChain生态与模型接入.ipynb)
